# SHIELD — All financial predictors versus IG top 20

The original file has **97 columns**: `Company`, `Bankrupt`, and **95 financial predictors**.
`Company` is an identifier and `Bankrupt` is the target: neither is an input feature.
The all-feature arm retains **all 95 financial predictors**, including constant `Net Income Flag`,
and **does not apply IG**. The comparison arm follows the original cleaning and selects top 20 by IG.

Compare the same nine RF/XGBoost candidates using the same stratified **80:20 split**, seed 42,
and **5-fold training CV**. SMOTE is fitted only on training folds. Evaluate default 0.5 cutoffs and
thresholds selected using training out-of-fold predictions. Parameters are fixed; no large grid search.

**Question:** does removing the top-20 restriction improve the precision–recall trade-off,
particularly whether both can reach 70% together? The comparison uses matched companies and folds.
Models, the prototype input fields and risk bands are not changed automatically.

OOF scores are used for threshold/model selection and are therefore tuning scores. The historical
holdout has been examined before, so further results on it remain exploratory.


## 1. Dependencies
These modelling versions match the verified local environment; versions are recorded per run.

In [ ]:
%pip install -q scikit-learn==1.9.0 imbalanced-learn==0.14.2 xgboost==2.0.3 openpyxl matplotlib joblib


## 2. Paths
Use the **original Excel file**. A previously cleaned CSV might already have removed a financial predictor.
Do not use SMOTE-resampled or IG-selected arrays as input.


In [ ]:
from pathlib import Path
from IPython.display import Image, display
try:
    from google.colab import drive
    drive.mount('/content/drive')
    RAW_PATH=Path('/content/drive/MyDrive/SHIELD_Project/data/bankruptcy_italian_companies_2023.xlsx')
    OUTPUT_BASE=Path('/content/drive/MyDrive/SHIELD_Project/reports/feature_comparison')
except ImportError:
    PROJECT=Path.cwd()
    if PROJECT.name=='notebooks':
        PROJECT=PROJECT.parent
    RAW_PATH=PROJECT/'data'/'bankruptcy_italian_companies_2023.xlsx'
    OUTPUT_BASE=PROJECT/'reports'/'feature_comparison'

# Default: run both matched arms. Optionally reuse a completed IG run only if its
# data fingerprint, model settings and package versions match exactly.
PRIOR_IG_DIR=None
print('Input:',RAW_PATH)
print('Reports:',OUTPUT_BASE)


## 3. Experiment runner
Embedded here so no helper Python files need uploading to Colab.

In [ ]:
"""Training-fold threshold experiments for SHIELD; also embedded in the Colab notebook."""
from __future__ import annotations

import argparse
import hashlib
import importlib.metadata
import json
import platform
import time
from dataclasses import asdict, dataclass
from functools import partial
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.metrics import (
    accuracy_score, average_precision_score, classification_report, confusion_matrix,
    f1_score, fbeta_score, precision_recall_curve, precision_score, recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from xgboost import XGBClassifier


@dataclass(frozen=True)
class Config:
    seed: int = 42
    folds: int = 5
    test_size: float = 0.20
    top_k: int = 20
    smote_neighbors: int = 5
    target_precision: float = 0.70
    target_recall: float = 0.70
    model_jobs: int = 2
    include_leaf_variants: bool = True
    evaluate_holdout: bool = True
    feature_mode: str = "ig20"  # ig20: top-k IG; all: every financial predictor, no IG
    version: str = "shield-focused-threshold-v2"


def candidate_specs(include_leaf_variants=True):
    rf = dict(n_estimators=200, max_depth=10, min_samples_split=2,
              min_samples_leaf=1, max_features="sqrt", class_weight=None)
    xgb = dict(n_estimators=100, max_depth=6, learning_rate=0.1)
    specs = [
        dict(id="RF_SMOTE025", model="RandomForest", smote=0.25, params=rf.copy()),
        dict(id="RF_SMOTE050", model="RandomForest", smote=0.50,
             params={**rf, "min_samples_split": 5}),
        dict(id="XGB_SMOTE010", model="XGBoost", smote=0.10, params=xgb.copy()),
        dict(id="XGB_SMOTE050", model="XGBoost", smote=0.50, params=xgb.copy()),
        dict(id="RF_NO_SMOTE", model="RandomForest", smote=None, params=rf.copy()),
        dict(id="RF_NO_SMOTE_WEIGHTED", model="RandomForest", smote=None,
             params={**rf, "class_weight": "balanced_subsample"}),
        dict(id="XGB_NO_SMOTE", model="XGBoost", smote=None, params=xgb.copy()),
    ]
    if include_leaf_variants:
        specs += [
            dict(id="RF_SMOTE025_LEAF3", model="RandomForest", smote=0.25,
                 params={**rf, "min_samples_leaf": 3}),
            dict(id="RF_SMOTE025_LEAF3_FEATURES050", model="RandomForest", smote=0.25,
                 params={**rf, "min_samples_leaf": 3, "max_features": 0.5}),
        ]
    return specs


def clean_data(raw, top_k, retain_constants=False):
    df = raw.copy()
    if "Bankrupt" not in df:
        raise ValueError("Input must have Bankrupt: 0=Not Bankrupt, 1=Bankrupt.")
    if "Company" in df and df["Company"].duplicated().any():
        raise ValueError("Repeated Company IDs: resolve/group them before splitting.")
    df = df.drop(columns=["Company"], errors="ignore")
    if not retain_constants:
        df = df.drop(columns=[c for c in df if c != "Bankrupt" and
                              df[c].nunique(dropna=False) <= 1])
    if df.isna().any().any():
        raise ValueError("Missing data: investigate before running this comparison.")
    if set(df["Bankrupt"].unique()) != {0, 1}:
        raise ValueError("Bankrupt must contain exactly the labels 0 and 1.")
    X = df.drop(columns="Bankrupt")
    if len(X.columns) < top_k or not all(pd.api.types.is_numeric_dtype(X[c]) for c in X):
        raise ValueError("Supply at least top_k numeric financial predictors, before IG/SMOTE.")
    if not np.isfinite(X.to_numpy(dtype=float)).all():
        raise ValueError("Infinite values in input.")
    if X.duplicated().any():
        raise ValueError("Repeated predictor rows: investigate/group these before CV.")
    y = df["Bankrupt"].astype(int)
    if y.value_counts()[1] >= y.value_counts()[0]:
        raise ValueError("This experiment expects Bankrupt=1 to be the minority class.")
    return X, y


def make_pipeline(spec, config, memory=None):
    if config.feature_mode not in {"ig20", "all"}:
        raise ValueError("feature_mode must be ig20 or all.")
    if spec["model"] == "RandomForest":
        model = RandomForestClassifier(**spec["params"], random_state=config.seed,
                                       n_jobs=config.model_jobs)
    else:
        model = XGBClassifier(**spec["params"], random_state=config.seed,
                              n_jobs=config.model_jobs, eval_metric="logloss")
    steps = []
    if config.feature_mode == "ig20":
        selector = SelectKBest(partial(mutual_info_classif, random_state=config.seed),
                               k=config.top_k).set_output(transform="pandas")
        steps.append(("ig", selector))
    if spec["smote"] is not None:
        steps.append(("smote", SMOTE(sampling_strategy=spec["smote"],
                                      random_state=config.seed,
                                      k_neighbors=config.smote_neighbors)))
    steps.append(("model", model))
    return Pipeline(steps, memory=memory)


def fitted_features(pipe, columns):
    if "ig" in pipe.named_steps:
        return columns[pipe.named_steps["ig"].get_support()].tolist()
    return list(columns)


def data_fingerprint(X, y):
    hasher = hashlib.sha256()
    hasher.update(pd.util.hash_pandas_object(X, index=True).values.tobytes())
    hasher.update(pd.util.hash_pandas_object(y, index=True).values.tobytes())
    hasher.update(json.dumps([(c, str(X[c].dtype)) for c in X]).encode())
    return hasher.hexdigest()


def metrics(y, scores, threshold):
    pred = (np.asarray(scores) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {
        "Accuracy (%)": accuracy_score(y, pred) * 100,
        "Precision (%)": precision_score(y, pred, zero_division=0) * 100,
        "Recall (%)": recall_score(y, pred, zero_division=0) * 100,
        "F1-Score (%)": f1_score(y, pred, zero_division=0) * 100,
        "ROC-AUC (%)": roc_auc_score(y, scores) * 100,
        "F2-Score (%)": fbeta_score(y, pred, beta=2, zero_division=0) * 100,
        "Average Precision (%)": average_precision_score(y, scores) * 100,
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
    }


def threshold_table(y, scores):
    """Exact observed thresholds; equal scores are always kept in the same group."""
    y, scores = np.asarray(y), np.asarray(scores)
    p, r, thresholds = precision_recall_curve(y, scores)
    p, r = p[:-1], r[:-1]  # final (precision=1, recall=0) has no threshold
    tp = np.rint(r * np.sum(y == 1)).astype(int)
    positives = np.searchsorted(np.sort(scores), thresholds, side="left")
    fp = len(scores) - positives - tp
    fn = int(np.sum(y == 1)) - tp
    tn = int(np.sum(y == 0)) - fp
    f1 = np.divide(2*p*r, p+r, out=np.zeros_like(p), where=(p+r) > 0)
    f2 = np.divide(5*p*r, 4*p+r, out=np.zeros_like(p), where=(4*p+r) > 0)
    return pd.DataFrame({"Threshold": thresholds, "Precision": p, "Recall": r,
                         "F1": f1, "F2": f2, "TP": tp, "FP": fp, "FN": fn, "TN": tn})


def select_threshold(table, config):
    """First try both 70% targets, otherwise maximize F2 subject to the recall floor."""
    recall_ok = table[table.Recall >= config.target_recall - 1e-12]
    both_ok = recall_ok[recall_ok.Precision >= config.target_precision - 1e-12]
    pool = both_ok if len(both_ok) else recall_ok
    if pool.empty:
        raise ValueError("No threshold satisfies the configured recall floor.")
    selected = pool.sort_values(["F2", "Precision", "Recall", "Threshold"],
                                ascending=[False, False, False, False]).iloc[0]
    return float(selected.Threshold), bool(len(both_ok))


def wilson_interval(successes, total):
    if total == 0:
        return None
    z = 1.959963984540054
    p = successes / total
    denom = 1 + z*z/total
    center = (p + z*z/(2*total))/denom
    margin = z*np.sqrt(p*(1-p)/total + z*z/(4*total*total))/denom
    return [float(100*(center-margin)), float(100*(center+margin))]


def _write_json(path, value):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(value, indent=2, allow_nan=False), encoding="utf-8")
    tmp.replace(path)


def run_focus(raw, output_base, config=Config(), source_note="", specs=None):
    """Tune from OOF training predictions. Freeze choices before touching holdout labels."""
    specs = candidate_specs(config.include_leaf_variants) if specs is None else specs
    if not specs or len({s["id"] for s in specs}) != len(specs):
        raise ValueError("Supply distinct candidate IDs.")
    if "RF_SMOTE025" not in {s["id"] for s in specs}:
        raise ValueError("Include RF_SMOTE025 as the reference baseline.")
    if config.feature_mode not in {"ig20", "all"}:
        raise ValueError("feature_mode must be ig20 or all.")
    X, y = clean_data(raw, config.top_k, retain_constants=config.feature_mode == "all")
    train_idx, test_idx = train_test_split(np.arange(len(y)), test_size=config.test_size,
                                          random_state=config.seed, stratify=y)
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_test, y_test = X.iloc[test_idx], y.iloc[test_idx]
    cv = list(StratifiedKFold(config.folds, shuffle=True, random_state=config.seed)
              .split(X_train, y_train))
    for a, b in cv:
        if len(np.intersect1d(a, b)) or y_train.iloc[b].nunique() != 2:
            raise ValueError("Invalid validation fold.")
        counts = y_train.iloc[a].value_counts()
        for spec in specs:
            if spec["smote"] is not None and (
                counts[1] <= config.smote_neighbors or
                int(spec["smote"] * counts[0]) <= counts[1]
            ):
                raise ValueError("SMOTE is not feasible for this data/fold/configuration.")
    versions = {name: importlib.metadata.version(name) for name in
                ["numpy", "pandas", "scikit-learn", "imbalanced-learn", "xgboost", "joblib"]}
    run_spec = dict(config=asdict(config), candidates=specs, packages=versions,
                    python=platform.python_version(), data_fingerprint=data_fingerprint(X, y),
                    source_note=source_note, rows=len(y), predictors=len(X.columns),
                    labels=y.value_counts().sort_index().to_dict(),
                    selection_policy="both targets if feasible; otherwise max F2 with recall >= target",
                    interpretation="OOF threshold/candidate-selection scores are tuning scores; holdout was historically reused")
    run_id = hashlib.sha256(json.dumps(run_spec, sort_keys=True).encode()).hexdigest()[:16]
    out = Path(output_base) / run_id
    out.mkdir(parents=True, exist_ok=True)
    _write_json(out / "run_config.json", run_spec)
    membership = pd.DataFrame({"Source_Row": train_idx, "Partition": "training",
                               "Fold": np.zeros(len(train_idx), dtype=int)})
    for fold, (_, b) in enumerate(cv, 1):
        membership.loc[b, "Fold"] = fold
    membership = pd.concat([membership, pd.DataFrame({"Source_Row": test_idx,
                            "Partition": "holdout", "Fold": -1})], ignore_index=True)
    membership.to_csv(out / "partition_membership.csv", index=False)
    memory = joblib.Memory(out / "pipeline_cache", verbose=0)
    summary_rows, threshold_frames, fold_rows = [], {}, []
    print(f"Run {run_id}: {len(specs)} candidates x {config.folds} training folds", flush=True)
    print(f"Feature mode: {config.feature_mode}; input predictors: {len(X.columns)}. "
          "SMOTE fit on training folds only; thresholds use OOF training predictions.", flush=True)
    print("OOF selection metrics are exploratory tuning results, not an unbiased final estimate.", flush=True)
    for spec in specs:
        scores = np.full(len(y_train), np.nan)
        assigned = np.zeros(len(y_train), dtype=int)
        for fold, (a, b) in enumerate(cv, 1):
            cp = out / f"{spec['id']}_fold{fold}.npz"
            detail_path = out / f"{spec['id']}_fold{fold}.json"
            if cp.exists() and detail_path.exists():
                saved = np.load(cp, allow_pickle=False)
                np.testing.assert_array_equal(saved["validation_positions"], b)
                fold_scores = saved["scores"]
                detail = json.loads(detail_path.read_text(encoding="utf-8"))
            else:
                start = time.perf_counter()
                pipe = make_pipeline(spec, config, memory)
                pipe.fit(X_train.iloc[a], y_train.iloc[a])
                positive_column = list(pipe.classes_).index(1)
                fold_scores = pipe.predict_proba(X_train.iloc[b])[:, positive_column]
                chosen = fitted_features(pipe, X.columns)
                detail = dict(Candidate=spec["id"], Fold=fold, Training_Count=len(a),
                              Validation_Count=len(b), Selected_Features=chosen,
                              Elapsed_Seconds=time.perf_counter()-start,
                              **metrics(y_train.iloc[b], fold_scores, 0.5))
                np.savez_compressed(cp, validation_positions=b, scores=fold_scores)
                _write_json(detail_path, detail)
            scores[b] = fold_scores
            assigned[b] += 1
            fold_rows.append(detail)
            print(f"  {spec['id']} fold {fold}/{config.folds} complete", flush=True)
        assert np.all(assigned == 1) and np.isfinite(scores).all()
        pd.DataFrame({"Source_Row": train_idx, "True_Label": y_train.to_numpy(),
                      "OOF_Score": scores, "Fold": membership.iloc[:len(train_idx)].Fold.to_numpy()
                      }).to_csv(out / f"{spec['id']}_oof_predictions.csv", index=False)
        table = threshold_table(y_train, scores)
        table.to_csv(out / f"{spec['id']}_thresholds.csv", index=False)
        threshold_frames[spec["id"]] = table
        threshold, feasible = select_threshold(table, config)
        chosen_metrics = metrics(y_train, scores, threshold)
        for rule, t in [("Default 0.5", 0.5), ("OOF selected", threshold)]:
            row = dict(Model=spec["model"], Experiment="80-20 + training CV threshold study",
                       Split=config.test_size, CV=config.folds, Candidate=spec["id"],
                       SMOTE_Ratio=spec["smote"], Decision_Rule=rule, Threshold=t,
                       Feature_Mode=config.feature_mode,
                       Feature_Count=config.top_k if config.feature_mode == "ig20" else len(X.columns),
                       Best_Params=spec["params"], Parameter_Source="predeclared fixed candidate",
                       Evaluation="training OOF tuning; not independent test",
                       **metrics(y_train, scores, t))
            row["CV_F1_Mean (%)"] = float(np.mean([
                f1_score(y_train.iloc[b], scores[b] >= t, zero_division=0)*100 for _, b in cv]))
            row["Both_Targets_Feasible_OOF"] = feasible
            recall_pool = table[table.Recall >= config.target_recall - 1e-12]
            row["Max_Precision_At_Target_Recall (%)"] = float(recall_pool.Precision.max()*100)
            summary_rows.append(row)
        print(f"  OOF cutoff {threshold:.6f}: P={chosen_metrics['Precision (%)']:.2f}% "
              f"R={chosen_metrics['Recall (%)']:.2f}% F2={chosen_metrics['F2-Score (%)']:.2f}% "
              f"both targets={feasible}", flush=True)
        pd.DataFrame(summary_rows).to_csv(out / "training_oof_results.csv", index=False)
    results = pd.DataFrame(summary_rows)
    selected = results[results.Decision_Rule == "OOF selected"]
    winner = selected.sort_values(["Both_Targets_Feasible_OOF", "F2-Score (%)", "Precision (%)", "Candidate"],
                                  ascending=[False, False, False, True]).iloc[0]
    choice = dict(Candidate=winner.Candidate, Threshold=float(winner.Threshold),
                  Selection_Data="training OOF only", Both_Targets_Feasible_OOF=bool(winner.Both_Targets_Feasible_OOF),
                  Holdout_Status="not evaluated yet", Run_ID=run_id)
    _write_json(out / "frozen_choice.json", choice)  # freeze before predicting holdout
    pd.DataFrame(fold_rows).to_json(out / "fold_details.json", orient="records", indent=2)
    plot_training_curves(results, threshold_frames, out, config)
    test_rows = []
    if config.evaluate_holdout:
        ids = list(dict.fromkeys(["RF_SMOTE025", winner.Candidate]))
        specs_by_id = {s["id"]: s for s in specs}
        for candidate in ids:
            spec = specs_by_id[candidate]
            pipe = make_pipeline(spec, config, memory)
            pipe.fit(X_train, y_train)
            score = pipe.predict_proba(X_test)[:, list(pipe.classes_).index(1)]
            rules = [("Reference default 0.5", 0.5)] if candidate == "RF_SMOTE025" else []
            if candidate == winner.Candidate:
                if candidate != "RF_SMOTE025":
                    rules.append(("Selected candidate default 0.5", 0.5))
                rules.append(("Frozen OOF threshold", float(winner.Threshold)))
                joblib.dump(pipe, out / "selected_training_pipeline.joblib")
                _write_json(out / "selected_feature_manifest.json", {
                    "Candidate": candidate, "Threshold": float(winner.Threshold),
                    "Features": fitted_features(pipe, X.columns),
                    "Feature_Mode": config.feature_mode,
                    "All_Input_Columns": X.columns.tolist(),
                    "Score_Meaning": "uncalibrated bankruptcy model score; not validated risk-band probability",
                    "Production_Status": "research candidate; prototype unchanged"})
            prediction_data = pd.DataFrame({"Source_Row": test_idx, "True_Label": y_test.to_numpy(),
                                            "Score": score})
            for rule, threshold in rules:
                m = metrics(y_test, score, threshold)
                row = dict(Model=spec["model"], Experiment="80-20 + CV=5 threshold study",
                           Split=config.test_size, CV=config.folds, Candidate=candidate,
                           Decision_Rule=rule, Threshold=threshold, Best_Params=spec["params"],
                           Feature_Mode=config.feature_mode,
                           Feature_Count=config.top_k if config.feature_mode == "ig20" else len(X.columns),
                           **m, **{"CV_F1_Mean (%)": None},
                           Evaluation="historically reused 80:20 holdout; exploratory",
                           Precision_95CI=wilson_interval(m["TP"], m["TP"]+m["FP"]),
                           Recall_95CI=wilson_interval(m["TP"], m["TP"]+m["FN"]))
                test_rows.append(row)
                prediction_data[rule] = (score >= threshold).astype(int)
                print(f"\n=== Classification Report: {spec['model']} | {rule} ===")
                print(classification_report(y_test, score >= threshold, labels=[0, 1],
                      target_names=["Not Bankrupt", "Bankrupt"], zero_division=0))
                print(f"Accuracy: {m['Accuracy (%)']:.2f}% | Precision: {m['Precision (%)']:.2f}% | "
                      f"Recall: {m['Recall (%)']:.2f}% | F1: {m['F1-Score (%)']:.2f}% | "
                      f"ROC-AUC: {m['ROC-AUC (%)']:.2f}%", flush=True)
            prediction_data.to_csv(out / f"{candidate}_holdout_predictions.csv", index=False)
        choice["Holdout_Status"] = "evaluated after freeze; historically reused and exploratory"
        _write_json(out / "frozen_choice.json", choice)
        pd.DataFrame(test_rows).to_csv(out / "holdout_results.csv", index=False)
        _write_json(out / "holdout_results.json", test_rows)
    _write_json(out / "training_oof_results.json", summary_rows)
    write_report(out, config, choice, selected, test_rows, source_note)
    print(f"\nSaved reports: {out}", flush=True)
    return dict(output_dir=out, training_results=results, thresholds=threshold_frames,
                choice=choice, holdout_results=pd.DataFrame(test_rows))


def plot_training_curves(results, tables, out, config):
    import matplotlib.pyplot as plt
    selected = results[results.Decision_Rule == "OOF selected"]
    fig, ax = plt.subplots(figsize=(10, 7))
    for _, row in selected.iterrows():
        table = tables[row.Candidate]
        ax.plot(table.Recall*100, table.Precision*100, label=row.Candidate, linewidth=1.3)
    ax.axvline(config.target_recall*100, color="black", linestyle="--", linewidth=1)
    ax.axhline(config.target_precision*100, color="black", linestyle="--", linewidth=1)
    ax.set(xlabel="Bankrupt-class recall (%)", ylabel="Bankrupt-class precision (%)",
           xlim=(0, 100), ylim=(0, 100),
           title=f"Training out-of-fold precision–recall curves ({config.feature_mode}, {config.folds}-fold CV)")
    ax.legend(fontsize=8, loc="upper right")
    fig.text(.01, .01, "Source: training OOF predictions; candidate/threshold tuning, not independent evaluation.", fontsize=8)
    fig.tight_layout(rect=(0, .035, 1, 1))
    fig.savefig(out / "training_precision_recall.png", dpi=180)
    plt.close(fig)
    for _, row in selected.iterrows():
        table = tables[row.Candidate]
        fig, ax = plt.subplots(figsize=(9, 5))
        for metric in ["Precision", "Recall", "F1", "F2"]:
            ax.plot(table.Threshold, table[metric]*100, label=metric)
        ax.axvline(row.Threshold, color="black", linestyle="--", label=f"Selected cutoff {row.Threshold:.3f}")
        ax.axhline(70, color="gray", linestyle=":", linewidth=1)
        ax.set(xlabel="Uncalibrated bankruptcy score cutoff (0–1)", ylabel="Bankrupt-class score (%)",
               ylim=(0, 100), title=f"{row.Candidate}: metrics across training OOF thresholds")
        ax.legend(fontsize=8)
        fig.text(.01, .01, "Source: training OOF predictions; scores shown are used for threshold selection.", fontsize=8)
        fig.tight_layout(rect=(0, .035, 1, 1))
        fig.savefig(out / f"{row.Candidate}_threshold_curve.png", dpi=160)
        plt.close(fig)


def write_report(out, config, choice, selected, test_rows, source_note):
    any_joint_target = bool(selected.Both_Targets_Feasible_OOF.any())
    best_precision = float(selected["Max_Precision_At_Target_Recall (%)"].max())
    lines = ["SHIELD - FOCUSED IMPROVEMENT ROUND", "",
             f"OUTCOME: Both precision/recall targets available on training OOF curves: {any_joint_target}.",
             f"Highest OOF precision with recall >= {config.target_recall*100:.0f}%: {best_precision:.2f}%.",
             "This round does not establish a permanent limit for the dataset.", "",
             f"Run ID: {choice['Run_ID']}",
             f"Source: {source_note}", f"Seed: {config.seed}; split: 80:20; training CV: {config.folds}",
             (f"IG: top {config.top_k}, fitted within every training fold." if config.feature_mode == "ig20"
              else "All financial predictors retained, including constants. No IG selection."),
             "Company identifier and Bankrupt target excluded from predictors. SMOTE: training folds only.",
             "Decision: score >= cutoff. Scores are not calibrated bankruptcy probabilities.",
             "All candidates use fixed, declared parameters; no grid search or nested CV.",
             "Threshold selection: both precision/recall targets if feasible; otherwise max F2 with recall floor.",
             "OOF results below are tuning scores after threshold/model selection, not independent performance.",
             "The old 80:20 holdout has already been used in historical experiments: further results are exploratory.",
             "A new split of these same companies cannot undo historical test reuse; external/new companies are preferable.",
             "", "TRAINING OOF THRESHOLD RESULTS", ""]
    for _, row in selected.iterrows():
        lines += [f"{row.Candidate} | cutoff {row.Threshold:.8f}",
                  f"  Precision: {row['Precision (%)']:.2f}% | Recall: {row['Recall (%)']:.2f}% | F1: {row['F1-Score (%)']:.2f}% | F2: {row['F2-Score (%)']:.2f}%",
                  f"  Both targets available: {row.Both_Targets_Feasible_OOF}; maximum precision at target recall: {row['Max_Precision_At_Target_Recall (%)']:.2f}%",
                  f"  Params: {json.dumps(row.Best_Params, sort_keys=True)}", ""]
    lines += [f"Frozen choice before holdout: {choice['Candidate']} | cutoff {choice['Threshold']:.8f}",
              "", "EXPLORATORY HOLDOUT RESULTS", ""]
    for row in test_rows:
        lines += [f"{row['Candidate']} | {row['Decision_Rule']} | cutoff {row['Threshold']:.8f}",
                  f"  Accuracy: {row['Accuracy (%)']:.2f}% | Precision: {row['Precision (%)']:.2f}% | Recall: {row['Recall (%)']:.2f}% | F1: {row['F1-Score (%)']:.2f}% | ROC-AUC: {row['ROC-AUC (%)']:.2f}%",
                  f"  TN={row['TN']}; FP={row['FP']}; FN={row['FN']}; TP={row['TP']}",
                  f"  Precision 95% Wilson interval: {row['Precision_95CI']}; Recall interval: {row['Recall_95CI']}", ""]
    lines += ["References:", "https://scikit-learn.org/stable/modules/classification_threshold.html",
              "https://imbalanced-learn.org/stable/common_pitfalls.html", ""]
    (out / "FOCUSED_RESULTS.txt").write_text("\n".join(lines), encoding="utf-8")



## 4. Matched comparison functions
Reused IG results must pass strict data/settings/package checks; fold membership is verified explicitly.

In [ ]:
"""Matched comparison of IG top 20 and all original financial predictors."""
from __future__ import annotations

import argparse
import hashlib
import importlib.metadata
import json
from dataclasses import asdict, replace
from pathlib import Path

import numpy as np
import pandas as pd

# Experiment functions were defined in the preceding cell.


def load_prior_ig(folder, raw, config, specs):
    """Only reuse the completed top-20 run if its data/settings/packages are identical."""
    folder = Path(folder)
    metadata = json.loads((folder / "run_config.json").read_text(encoding="utf-8"))
    X, y = clean_data(raw, config.top_k)
    if metadata["data_fingerprint"] != data_fingerprint(X, y):
        raise ValueError("Prior IG data differs. Omit prior_ig_dir to run a fresh matched comparison.")
    fields = ["seed", "folds", "test_size", "top_k", "smote_neighbors", "target_precision",
              "target_recall", "model_jobs", "include_leaf_variants", "evaluate_holdout"]
    for key in fields:
        if metadata["config"][key] != asdict(config)[key]:
            raise ValueError(f"Prior IG setting differs: {key}.")
    if metadata["config"].get("feature_mode", "ig20") != "ig20":
        raise ValueError("Prior folder must be the IG top-k run.")
    if metadata["candidates"] != specs:
        raise ValueError("Prior IG candidate parameters differ.")
    for name, version in metadata["packages"].items():
        if importlib.metadata.version(name) != version:
            raise ValueError(f"Prior IG package version differs: {name}.")
    results = pd.DataFrame(json.loads((folder / "training_oof_results.json").read_text(encoding="utf-8")))
    results["Feature_Mode"], results["Feature_Count"] = "ig20", config.top_k
    if len(results) != 2*len(specs) or results.Candidate.nunique() != len(specs):
        raise ValueError("Prior IG result set is incomplete.")
    thresholds = {s["id"]: pd.read_csv(folder / f"{s['id']}_thresholds.csv", float_precision="round_trip") for s in specs}
    for spec in specs:
        p = pd.read_csv(folder / f"{spec['id']}_oof_predictions.csv", float_precision="round_trip")
        if p.Source_Row.duplicated().any() or not np.isfinite(p.OOF_Score).all():
            raise ValueError("Prior IG OOF predictions are invalid.")
    holdout_path = folder / "holdout_results.json"
    holdout = pd.DataFrame(json.loads(holdout_path.read_text(encoding="utf-8"))) if holdout_path.exists() else pd.DataFrame()
    if len(holdout):
        holdout["Feature_Mode"], holdout["Feature_Count"] = "ig20", config.top_k
    return dict(output_dir=folder, training_results=results, thresholds=thresholds,
                choice=json.loads((folder / "frozen_choice.json").read_text(encoding="utf-8")),
                holdout_results=holdout)


def run_feature_comparison(raw, output_base, config=Config(), prior_ig_dir=None, source_note=""):
    specs = candidate_specs(config.include_leaf_variants)
    X_all, _ = clean_data(raw, config.top_k, retain_constants=True)
    constants = [c for c in X_all if X_all[c].nunique(dropna=False) <= 1]
    print(f"Raw columns: {len(raw.columns)}. Excluded Company and Bankrupt; "
          f"all-feature arm retains {len(X_all.columns)} numeric financial predictors.", flush=True)
    print(f"Retained constant predictors: {constants}", flush=True)
    ig_config = replace(config, feature_mode="ig20")
    if prior_ig_dir is not None:
        ig = load_prior_ig(prior_ig_dir, raw, ig_config, specs)
        print(f"Reusing verified matching IG run: {prior_ig_dir}", flush=True)
    else:
        ig = run_focus(raw, output_base, ig_config, source_note, specs)
    full = run_focus(raw, output_base, replace(config, feature_mode="all"), source_note, specs)
    old_members = pd.read_csv(ig["output_dir"] / "partition_membership.csv")
    new_members = pd.read_csv(full["output_dir"] / "partition_membership.csv")
    pd.testing.assert_frame_equal(old_members, new_members)
    for spec in specs:
        old = pd.read_csv(ig["output_dir"] / f"{spec['id']}_oof_predictions.csv", float_precision="round_trip")
        new = pd.read_csv(full["output_dir"] / f"{spec['id']}_oof_predictions.csv", float_precision="round_trip")
        pd.testing.assert_frame_equal(old[["Source_Row", "True_Label", "Fold"]],
                                      new[["Source_Row", "True_Label", "Fold"]])
    signature = {"ig_run": str(ig["output_dir"]), "all_run": str(full["output_dir"]),
                 "config": asdict(config), "comparison_version": "shield-all-features-v1"}
    compare_id = hashlib.sha256(json.dumps(signature, sort_keys=True).encode()).hexdigest()[:16]
    out = Path(output_base) / f"comparison_{compare_id}"
    out.mkdir(parents=True, exist_ok=True)
    (out / "comparison_config.json").write_text(json.dumps({
        **signature, "raw_columns": len(raw.columns), "all_financial_predictors": len(X_all.columns),
        "constant_predictors_retained": constants,
        "membership_verified": "identical companies, labels, train/holdout split and training CV folds",
        "source": source_note,
    }, indent=2), encoding="utf-8")
    metrics = ["Accuracy (%)", "Precision (%)", "Recall (%)", "F1-Score (%)", "ROC-AUC (%)",
               "F2-Score (%)", "Average Precision (%)", "Max_Precision_At_Target_Recall (%)"]
    keys = ["Candidate", "Decision_Rule"]
    paired = ig["training_results"].merge(full["training_results"], on=keys,
                                          suffixes=("_IG20", "_All"), validate="one_to_one")
    for metric in metrics:
        paired[metric + "_Change_pp"] = paired[metric + "_All"] - paired[metric + "_IG20"]
    paired.to_csv(out / "paired_training_comparison.csv", index=False)
    all_results = pd.concat([ig["training_results"], full["training_results"]], ignore_index=True)
    all_results.to_csv(out / "all_training_results.csv", index=False)
    holdouts = pd.concat([ig["holdout_results"], full["holdout_results"]], ignore_index=True)
    if len(holdouts):
        holdouts.to_csv(out / "exploratory_holdout_comparison.csv", index=False)
    selected = all_results[all_results.Decision_Rule == "OOF selected"]
    winner = selected.sort_values(["Both_Targets_Feasible_OOF", "F2-Score (%)", "Precision (%)", "Candidate"],
                                  ascending=[False, False, False, True]).iloc[0]
    comparison_choice = dict(Feature_Mode=winner.Feature_Mode, Candidate=winner.Candidate,
                              Threshold=float(winner.Threshold),
                              Selection_Data="training OOF only; historical holdouts not used for selection")
    (out / "training_selected_comparison_choice.json").write_text(
        json.dumps(comparison_choice, indent=2), encoding="utf-8")
    plot_comparison(paired, ig, full, out, config)
    write_comparison_report(out, ig, full, paired, config, source_note, len(X_all.columns), constants)
    print(f"Comparison saved: {out}", flush=True)
    return dict(output_dir=out, ig=ig, full=full, paired=paired,
                training_results=all_results, holdout_results=holdouts, choice=comparison_choice)


def plot_comparison(paired, ig, full, out, config):
    import matplotlib.pyplot as plt
    rows = paired[paired.Decision_Rule == "OOF selected"].reset_index(drop=True)
    metric = "Max_Precision_At_Target_Recall (%)"
    fig, ax = plt.subplots(figsize=(11, 6))
    x = np.arange(len(rows))
    ax.barh(x-.18, rows[metric+"_IG20"], height=.36, label=f"IG top {config.top_k}", color="#4878a8")
    ax.barh(x+.18, rows[metric+"_All"], height=.36, label="All financial predictors", color="#777777")
    ax.axvline(config.target_precision*100, color="black", linestyle="--", label="70% precision target")
    ax.set(yticks=x, yticklabels=rows.Candidate, xlim=(0,100),
           xlabel=f"Maximum bankruptcy precision while recall >= {config.target_recall*100:.0f}% (%)",
           ylabel="Fixed model/SMOTE candidate",
           title=f"All financial predictors versus IG top {config.top_k}: training OOF threshold potential")
    ax.invert_yaxis()
    ax.legend(fontsize=8)
    fig.text(.01,.01,"Source: matched 5-fold training OOF curves. Tuning results; not independent final evaluation.",fontsize=8)
    fig.tight_layout(rect=(0,.04,1,1))
    fig.savefig(out/"precision_at_70_recall_comparison.png",dpi=180)
    plt.close(fig)
    winner_ids = list(dict.fromkeys(["RF_SMOTE025", ig["choice"]["Candidate"], full["choice"]["Candidate"]]))
    fig, axes = plt.subplots(1,len(winner_ids),figsize=(6*len(winner_ids),5.5),squeeze=False)
    for ax, candidate in zip(axes.flat,winner_ids):
        for run,label,color in [(ig,f"IG top {config.top_k}","#4878a8"),
                                 (full,"All financial predictors","#777777")]:
            table=run["thresholds"][candidate]
            ax.plot(table.Recall*100,table.Precision*100,label=label,color=color)
        ax.axvline(config.target_recall*100,color="black",linestyle="--",linewidth=1)
        ax.axhline(config.target_precision*100,color="black",linestyle="--",linewidth=1)
        ax.set(xlim=(0,100),ylim=(0,100),xlabel="Bankruptcy recall (%)",ylabel="Bankruptcy precision (%)",title=candidate)
        ax.legend(fontsize=8)
    fig.suptitle("Matched training OOF precision–recall curves",fontsize=14)
    fig.text(.01,.01,"Source: same companies, folds and declared model settings; thresholds selected on training OOF predictions.",fontsize=8)
    fig.tight_layout(rect=(0,.04,1,.95))
    fig.savefig(out/"matched_precision_recall_curves.png",dpi=180)
    plt.close(fig)


def write_comparison_report(out, ig, full, paired, config, source_note, full_count, constants):
    ig_rows=ig["training_results"].query("Decision_Rule == 'OOF selected'")
    full_rows=full["training_results"].query("Decision_Rule == 'OOF selected'")
    metric="Max_Precision_At_Target_Recall (%)"
    lines=["SHIELD - ALL FINANCIAL PREDICTORS VERSUS IG TOP 20", "",
           f"Top-{config.top_k} IG: any candidate reaches both targets: {bool(ig_rows.Both_Targets_Feasible_OOF.any())}.",
           f"All {full_count} financial predictors: any candidate reaches both targets: {bool(full_rows.Both_Targets_Feasible_OOF.any())}.",
           f"Maximum OOF precision at recall >=70%: IG={ig_rows[metric].max():.2f}%; All={full_rows[metric].max():.2f}%.", "",
           f"Source: {source_note}", "Raw file has 97 columns, not 97 financial predictors.",
           "Company is an identifier; Bankrupt is the target. Both are excluded from predictors.",
           f"All-feature arm: {full_count} financial predictors, no IG. Constants retained: {constants}.",
           "IG arm: previous cleaning removes constants, then training-fold IG selects top 20.",
           "Same seed=42, 80:20 split, five training CV folds, nine fixed candidates and threshold-selection rule.",
           "SMOTE remains training-fold only. No new models, parameter grids, or nested CV.",
           "Changing the feature count also changes the meaning of fractional/sqrt max_features in RF.",
           "OOF threshold/model-selection metrics are tuning scores. Reused historical holdout scores are exploratory.",
           f"IG report: {ig['output_dir']}", f"All-feature report: {full['output_dir']}", "",
           "MATCHED TRAINING OOF RESULTS - SELECTED CUTOFFS", ""]
    reference_lines = ["MATCHED RF SMOTE 0.25 AT CUTOFF 0.5 - EXPLORATORY HOLDOUT", ""]
    for label, run in [("IG top 20", ig), ("All financial predictors", full)]:
        reference = run["holdout_results"]
        if len(reference):
            reference = reference[reference.Decision_Rule == "Reference default 0.5"]
            for _, r in reference.iterrows():
                reference_lines += [f"{label}: Precision {r['Precision (%)']:.2f}% | Recall {r['Recall (%)']:.2f}% | F1 {r['F1-Score (%)']:.2f}% | TP={r.TP}; FP={r.FP}"]
    if len(reference_lines) > 2:
        lines[6:6] = reference_lines + [""]
    for _,r in paired[paired.Decision_Rule=="OOF selected"].iterrows():
        lines += [r.Candidate,
                  f"  IG20: P={r['Precision (%)_IG20']:.2f}% | R={r['Recall (%)_IG20']:.2f}% | F1={r['F1-Score (%)_IG20']:.2f}% | F2={r['F2-Score (%)_IG20']:.2f}% | cutoff={r.Threshold_IG20:.8f}",
                  f"  All:  P={r['Precision (%)_All']:.2f}% | R={r['Recall (%)_All']:.2f}% | F1={r['F1-Score (%)_All']:.2f}% | F2={r['F2-Score (%)_All']:.2f}% | cutoff={r.Threshold_All:.8f}",
                  f"  Max precision at recall >=70%: IG={r[metric+'_IG20']:.2f}%; All={r[metric+'_All']:.2f}%.",""]
    lines += ["EXPLORATORY HOLDOUT RESULTS - PER-ARM TRAINING-SELECTED CHOICES", ""]
    for label,run in [("IG top 20",ig),("All financial predictors",full)]:
        for _,r in run["holdout_results"].iterrows():
            lines += [f"{label} | {r.Candidate} | {r.Decision_Rule} | cutoff {r.Threshold:.8f}",
                      f"  Accuracy: {r['Accuracy (%)']:.2f}% | Precision: {r['Precision (%)']:.2f}% | Recall: {r['Recall (%)']:.2f}% | F1: {r['F1-Score (%)']:.2f}% | ROC-AUC: {r['ROC-AUC (%)']:.2f}%",
                      f"  TP={r.TP}; FP={r.FP}; FN={r.FN}; TN={r.TN}",""]
    lines += ["No production model, frontend fields or risk bands were changed.",
              "All comparison metrics and parameter settings are retained at full precision in the CSV files.",
              "Reload score/threshold CSVs using pandas.read_csv(..., float_precision='round_trip') to preserve equal-cutoff decisions.",""]
    (out/"FEATURE_COMPARISON_RESULTS.txt").write_text("\n".join(lines),encoding="utf-8")



## 5. Fixed settings and data checks
Threshold selection first looks for both precision and recall >=70%. If unavailable, it maximizes F2
subject to recall >=70%. This fallback does **not** achieve the joint target.
Every candidate is tested at default 0.5 as well, which lets you compare the feature sets without changing the cutoff.

RF `max_features='sqrt'` or `0.5` naturally considers a different number of inputs when the feature count changes.
The parameter setting remains fixed between arms. No additional scaling is introduced in this comparison.


In [ ]:
CONFIG=Config(seed=42,folds=5,test_size=.20,top_k=20,smote_neighbors=5,
              target_precision=.70,target_recall=.70,model_jobs=2,
              include_leaf_variants=True,evaluate_holdout=True)
RAW=pd.read_excel(RAW_PATH) if RAW_PATH.suffix.lower() in ['.xlsx','.xls'] else pd.read_csv(RAW_PATH)
X_ALL,Y=clean_data(RAW,CONFIG.top_k,retain_constants=True)
X_IG,_=clean_data(RAW,CONFIG.top_k)
print('Raw table:',RAW.shape)
print('All-financial-predictor arm:',X_ALL.shape)
print('IG arm inputs before selection:',X_IG.shape)
print('Original class counts:',Y.value_counts().sort_index().to_dict())
print('Retained constants:',[c for c in X_ALL if X_ALL[c].nunique(dropna=False)<=1])
assert 'Company' not in X_ALL and 'Bankrupt' not in X_ALL
display(pd.DataFrame(candidate_specs(CONFIG.include_leaf_variants)))


## 6. Run both arms
There are 45 training-fold fits per arm. Completed fold checkpoints can resume.
Without a prior IG run this is 90 fits plus limited holdout refits.
Feature mode and the data fingerprint isolate checkpoints, preventing top-20 predictions from
being reused as all-feature predictions. Each original training company receives one held-out-fold score.

Thresholds and per-arm candidate choices use training OOF predictions. The historical holdout is
evaluated after each arm freezes its choice. It is not used to select the comparison winner.


In [ ]:
COMPARISON=run_feature_comparison(RAW,OUTPUT_BASE,CONFIG,prior_ig_dir=PRIOR_IG_DIR,
                                  source_note=str(RAW_PATH))
REPORT_DIR=COMPARISON['output_dir']
print('Training-selected comparison choice:',COMPARISON['choice'])


## 7. Metrics in the existing report layout
Precision, recall, F1 and F2 refer to `Bankrupt (1)`. Tables keep the original headline metrics,
with feature mode, candidate, decision rule and threshold added for traceability.
Selected OOF metrics are tuning results. Mean fold F1 at the selected pooled cutoff is not independent evaluation.


In [ ]:
HEADLINE=['Model','Experiment','Split','CV','Accuracy (%)','Precision (%)','Recall (%)',
          'F1-Score (%)','ROC-AUC (%)','Best_Params','CV_F1_Mean (%)']
EXTRA=['Feature_Mode','Feature_Count','Candidate','SMOTE_Ratio','Decision_Rule','Threshold',
       'F2-Score (%)','Both_Targets_Feasible_OOF','Max_Precision_At_Target_Recall (%)',
       'TP','FP','FN','TN','Evaluation']
display(COMPARISON['training_results'][HEADLINE+EXTRA])
print('Any all-feature candidate reaches both 70% targets on its training OOF curve:',
      bool(COMPARISON['full']['training_results'].Both_Targets_Feasible_OOF.any()))


## 8. Matched comparisons
`Change_pp` columns are **percentage-point changes**, All minus IG20.
At a fixed cutoff the comparison isolates the input-feature policy. At selected cutoffs it compares
the complete feature-and-threshold policies. Both use exactly the same held-out training companies.


In [ ]:
PAIRED_COLUMNS=['Candidate','Decision_Rule','Precision (%)_IG20','Precision (%)_All',
                'Precision (%)_Change_pp','Recall (%)_IG20','Recall (%)_All','Recall (%)_Change_pp',
                'F1-Score (%)_IG20','F1-Score (%)_All','F2-Score (%)_IG20','F2-Score (%)_All',
                'Max_Precision_At_Target_Recall (%)_IG20','Max_Precision_At_Target_Recall (%)_All']
display(COMPARISON['paired'][PAIRED_COLUMNS])
display(Image(filename=str(REPORT_DIR/'precision_at_70_recall_comparison.png')))
display(Image(filename=str(REPORT_DIR/'matched_precision_recall_curves.png')))


## 9. Exploratory historical holdout
The unchanged reference RF is reported at cutoff 0.5 in each arm. Each arm's training-selected
candidate is also reported at its frozen cutoff. When those selected candidates differ, comparing their
test rows does not isolate feature count alone; the matched candidate-level OOF comparisons do.

These test companies have been examined in previous rounds. Treat the scores as exploratory,
and do not adjust cutoffs using them. New/external evaluation is needed for a fresh final assessment.


In [ ]:
if len(COMPARISON['holdout_results']):
    display(COMPARISON['holdout_results'])
else:
    print('Holdout evaluation disabled by CONFIG.')


## 10. Save and interpret
The readable record is `FEATURE_COMPARISON_RESULTS.txt`; CSVs retain full precision and model parameters.
Reload saved score/threshold CSVs with `pd.read_csv(path, float_precision='round_trip')` to preserve
scores that exactly equal the cutoff. The saved JSON records and fold NPZ arrays also preserve precision.
The all-feature arm has its own model manifest with all financial input fields. This research export
does not replace the 20-field prototype: any deployment change requires a separate implementation decision.

If full features help consistently, the top-20 restriction warrants reconsideration.
If they do not, this does not prove a permanent dataset ceiling; it tells us that removing IG alone,
under the tested settings, was insufficient.


In [ ]:
print((REPORT_DIR/'FEATURE_COMPARISON_RESULTS.txt').read_text(encoding='utf-8'))
print('Saved comparison:',REPORT_DIR)
print('Production model, frontend fields and risk bands remain unchanged.')
